In [1]:
# Install clean dependencies without PyTorch conflicts
!pip install -q fastapi uvicorn groq pypdf fastembed faiss-cpu python-multipart
print("✅ Libraries installed successfully!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.1/324.1 kB 21.9 MB/s eta 0:00:00
✅ Libraries installed successfully!


In [2]:
import os
from pathlib import Path

# Define project directories
base_dir = Path("rag-agent")
app_dir = base_dir / "app"
rag_dir = app_dir / "rag"
data_dir = base_dir / "data"
uploads_dir = data_dir / "uploads"
index_dir = data_dir / "faiss_index"

# Create directories
for folder in [app_dir, rag_dir, uploads_dir, index_dir]:
    folder.mkdir(parents=True, exist_ok=True)
    print(f"Created: {folder}")

# Create mandatory initialization files
(app_dir / "__init__.py").touch()
(rag_dir / "__init__.py").touch()
print("✅ Project directory tree built flawlessly!")


Created: rag-agent/app
Created: rag-agent/app/rag
Created: rag-agent/data/uploads
Created: rag-agent/data/faiss_index
✅ Project directory tree built flawlessly!


In [3]:
%%writefile rag-agent/app/config.py
import os
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

# Light-weight, high-speed embedding engine (384 dimensions)
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5")

CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "800"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "120"))
TOP_K = int(os.getenv("TOP_K", "4"))

DATA_DIR = BASE_DIR / "data"
UPLOAD_DIR = DATA_DIR / "uploads"
INDEX_DIR = DATA_DIR / "faiss_index"

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)


Writing rag-agent/app/config.py


In [4]:
%%writefile rag-agent/app/system_prompt.py
SYSTEM_PROMPT = """You are a highly helpful and professional Business Analytics with AI track assistant.

Your job is to answer the user's questions accurately using only the provided document contexts.

CRITICAL ESCALATION POLICY:
Always respond warmly. If the answer cannot be found in the provided context, or if there is no context uploaded yet, you must refuse to answer and strictly refer the user to Olumatin Thomas on 07037613488. Do not make up answers outside the provided text.
"""
print("✅ System escalation prompt file saved!")


Writing rag-agent/app/system_prompt.py


In [5]:
%%writefile rag-agent/app/models.py
from pydantic import BaseModel
from typing import List, Optional

class ChatRequest(BaseModel):
    question: str
    top_k: Optional[int] = 4

class SourceChunk(BaseModel):
    text: str
    source: str
    chunk_id: int
    score: Optional[float] = None

class ChatResponse(BaseModel):
    answer: str
    sources: List[SourceChunk]

class UploadResponse(BaseModel):
    filename: str
    chunks_added: int
    total_chunks: int


Writing rag-agent/app/models.py


In [6]:
%%writefile rag-agent/app/rag/ingest.py
import pickle
import numpy as np
import faiss
from pypdf import PdfReader
from fastembed import TextEmbedding

from app import config

_embedder = None
_index = None
_metadata = []

INDEX_FILE = config.INDEX_DIR / "index.faiss"
META_FILE = config.INDEX_DIR / "metadata.pkl"


def get_embedder():
    global _embedder
    if _embedder is None:
        _embedder = TextEmbedding(model_name=config.EMBEDDING_MODEL)
    return _embedder


def _embed(texts):
    embedder = get_embedder()
    vecs = np.array(list(embedder.embed(texts)), dtype="float32")
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms[norms == 0] = 1e-9
    return vecs / norms


def load_index():
    global _index, _metadata
    if INDEX_FILE.exists() and META_FILE.exists():
        _index = faiss.read_index(str(INDEX_FILE))
        with open(META_FILE, "rb") as f:
            _metadata = pickle.load(f)
    else:
        # BAAI/bge-small-en-v1.5 has an explicit vector dimension of 384
        dim = 384
        _index = faiss.IndexFlatIP(dim)
        _metadata = []
    return _index, _metadata


def save_index():
    faiss.write_index(_index, str(INDEX_FILE))
    with open(META_FILE, "wb") as f:
        pickle.dump(_metadata, f)


def chunk_text(text, chunk_size=None, overlap=None):
    chunk_size = chunk_size or config.CHUNK_SIZE
    overlap = overlap or config.CHUNK_OVERLAP
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return [c.strip() for c in chunks if c.strip()]


def extract_pdf_text(path):
    reader = PdfReader(str(path))
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages)


def ingest_pdf(path, source_name):
    global _index, _metadata
    if _index is None:
        load_index()

    text = extract_pdf_text(path)
    chunks = chunk_text(text)
    if not chunks:
        return 0, len(_metadata)

    vectors = _embed(chunks)
    _index.add(vectors)

    start_id = len(_metadata)
    for i, chunk in enumerate(chunks):
        _metadata.append({
            "text": chunk,
            "source": source_name,
            "chunk_id": start_id + i,
        })

    save_index()
    return len(chunks), len(_metadata)


Writing rag-agent/app/rag/ingest.py


In [7]:
%%writefile rag-agent/app/rag/retriever.py
from app import config
from app.rag import ingest


def search(query, top_k=None):
    top_k = top_k or config.TOP_K

    if ingest._index is None or ingest._index.ntotal == 0:
        return []

    query_vec = ingest._embed([query])
    scores, indices = ingest._index.search(query_vec, top_k)

    results = []
    # Flat 1D loop mapping protects the server from matrix indexing failures
    for score_row, idx_row in zip(scores, indices):
        for score, idx in zip(score_row, idx_row):
            if idx == -1:
                continue
            meta = ingest._metadata[idx]
            results.append({
                "text": meta["text"],
                "source": meta["source"],
                "chunk_id": meta["chunk_id"],
                "score": float(score),
            })
    return results


Writing rag-agent/app/rag/retriever.py


In [8]:
%%writefile rag-agent/app/rag/llm.py
from groq import Groq

from app import config
from app.system_prompt import SYSTEM_PROMPT

_client = None


def get_client():
    global _client
    if _client is None:
        _client = Groq(api_key=config.GROQ_API_KEY)
    return _client


def build_context_block(chunks):
    if not chunks:
        return "No relevant context was found in the knowledge base."
    parts = []
    for c in chunks:
        parts.append(f"[Source: {c['source']} | chunk {c['chunk_id']}]\n{c['text']}")
    return "\n\n---\n\n".join(parts)


def generate_answer(question, chunks):
    context = build_context_block(chunks)
    client = get_client()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]

    completion = client.chat.completions.create(
        model=config.GROQ_MODEL,
        messages=messages,
        temperature=0.2,
    )
    # Picks the message string accurately from Pydantic structural components
    return completion.choices[0].message.content


Writing rag-agent/app/rag/llm.py


In [9]:
%%writefile rag-agent/app/main.py
import shutil
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import HTMLResponse

from app import config
from app.models import ChatRequest, ChatResponse, UploadResponse, SourceChunk
from app.rag import ingest, retriever, llm

app = FastAPI(title="RAG Agent Dashboard")


@app.on_event("startup")
def startup():
    ingest.load_index()


@app.get("/health")
def health():
    total = 0 if ingest._index is None else ingest._index.ntotal
    return {"status": "ok", "chunks_indexed": total}


@app.post("/upload", response_model=UploadResponse)
async def upload_pdf(file: UploadFile = File(...)):
    if not file.filename.lower().endswith(".pdf") and not file.filename.lower().endswith(".txt"):
        raise HTTPException(status_code=400, detail="Only PDF or TXT files are supported.")

    dest = config.UPLOAD_DIR / file.filename
    with open(dest, "wb") as f:
        shutil.copyfileobj(file.file, f)

    added, total = ingest.ingest_pdf(dest, file.filename)
    return UploadResponse(filename=file.filename, chunks_added=added, total_chunks=total)


@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    # ULTIMATE SAFEGUARD: If database index is completely empty, fire your customized rules instantly!
    if ingest._index is None or ingest._index.ntotal == 0:
        fallback_msg = "Always respond warmly, if you are unable to answer, refer the user to Olumatin Thomas on 07037613488"
        return ChatResponse(answer=fallback_msg, sources=[])

    try:
        chunks = retriever.search(req.question, req.top_k)
        # Secondary fallback if retriever yields empty results
        if not chunks:
            fallback_msg = "Always respond warmly, if you are unable to answer, refer the user to Olumatin Thomas on 07037613488"
            return ChatResponse(answer=fallback_msg, sources=[])

        answer = llm.generate_answer(req.question, chunks)
        sources = [SourceChunk(**c) for c in chunks]
        return ChatResponse(answer=answer, sources=sources)
    except Exception as e:
        emergency_msg = "Always respond warmly, if you are unable to answer, refer the user to Olumatin Thomas on 07037613488"
        return ChatResponse(answer=emergency_msg, sources=[])


@app.get("/", response_class=HTMLResponse)
def serve_ui():
    return """
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>RAG Agent Chat Dashboard</title>
        <link href="https://jsdelivr.net" rel="stylesheet">
        <style>
            body { background-color: #f4f6f9; font-family: 'Segoe UI', system-ui, sans-serif; }
            .chat-container { height: 450px; overflow-y: auto; background: white; border-radius: 10px; padding: 20px; box-shadow: inset 0 0 10px rgba(0,0,0,0.05); }
            .user-msg { background-color: #0d6efd; color: white; border-radius: 15px 15px 0 15px; padding: 10px 15px; margin: 5px 0; max-width: 75%; float: right; clear: both; }
            .agent-msg { background-color: #e9ecef; color: #212529; border-radius: 15px 15px 15px 0; padding: 10px 15px; margin: 5px 0; max-width: 75%; float: left; clear: both; }
            .sources-box { font-size: 0.8rem; color: #6c757d; margin-top: 5px; border-left: 2px solid #dee2e6; padding-left: 8px; }
            .card { border: none; box-shadow: 0 4px 6px rgba(0,0,0,0.05); border-radius: 12px; }
        </style>
    </head>
    <body>
        <div class="container py-5">
            <header class="pb-3 mb-4 border-bottom">
                <span class="fs-4 fw-bold text-dark">📚 Business Analytics RAG Assistant</span>
            </header>

            <div class="row g-4">
                <div class="col-md-4">
                    <div class="card p-4 bg-white mb-4">
                        <h5 class="fw-bold mb-3">📁 Upload Knowledge Base</h5>
                        <p class="text-muted small">Upload your course PDFs or business files here.</p>
                        <div class="mb-3">
                            <input class="form-control" type="file" id="pdfFile" accept=".pdf,.txt">
                        </div>
                        <button class="btn btn-dark w-100" onclick="uploadDocument()">Upload Document</button>
                        <div id="uploadStatus" class="mt-3 small"></div>
                    </div>
                </div>

                <div class="col-md-8">
                    <div class="card p-4 bg-white">
                        <h5 class="fw-bold mb-3">💬 Chat with Agent</h5>
                        <div class="chat-container border mb-3" id="chatWindow">
                            <div class="agent-msg">Hello! Ask me any questions about our Business Analytics with AI track.</div>
                        </div>
                        <div class="input-group">
                            <input type="text" id="userInput" class="form-control" placeholder="Type your inquiry here..." onkeypress="if(event.key === 'Enter') sendMessage()">
                            <button class="btn btn-primary" onclick="sendMessage()">Send</button>
                        </div>
                    </div>
                </div>
            </div>
        </div>

        <script>
            async function uploadDocument() {
                const fileInput = document.getElementById('pdfFile');
                const statusDiv = document.getElementById('uploadStatus');
                if (!fileInput.files || fileInput.files.length === 0) { statusDiv.innerHTML = '<span class="text-danger">Select a file first!</span>'; return; }

                const formData = new FormData();
                // FIX: Pick the explicit single element out of the input list array to prevent data corruption
                formData.append('file', fileInput.files[0]);
                statusDiv.innerHTML = '<div class="spinner-border spinner-border-sm text-primary"></div> Processing...';

                try {
                    const response = await fetch('/upload', { method: 'POST', body: formData });
                    const resData = await response.json();
                    if(response.ok) {
                        statusDiv.innerHTML = `<span class="text-success">Processed ${resData.filename} (${resData.chunks_added} chunks added).</span>`;
                    } else {
                        statusDiv.innerHTML = `<span class="text-danger">Error: ${resData.detail}</span>`;
                    }
                } catch(e) { statusDiv.innerHTML = '<span class="text-danger">Upload failed.</span>'; }
            }

            async function sendMessage() {
                const input = document.getElementById('userInput');
                const chatWindow = document.getElementById('chatWindow');
                const query = input.value.trim();
                if (!query) return;

                chatWindow.innerHTML += `<div class="user-msg">${query}</div>`;
                const userQuestion = query;
                input.value = '';
                chatWindow.scrollTop = chatWindow.scrollHeight;

                try {
                    const response = await fetch('/chat', {
                        method: 'POST',
                        headers: { 'Content-Type': 'application/json' },
                        body: JSON.stringify({ question: userQuestion })
                    });
                    const resData = await response.json();

                    let sourcesHtml = '';
                    if(resData.sources && resData.sources.length > 0) {
                        sourcesHtml = '<div class="sources-box"><strong>Sources:</strong> ' +
                            resData.sources.map(s => `${s.source} (id: ${s.chunk_id})`).join(', ') + '</div>';
                    }

                    chatWindow.innerHTML += `<div class="agent-msg"><div>${resData.answer}</div>${sourcesHtml}</div>`;
                } catch(e) {
                    chatWindow.innerHTML += `<div class="agent-msg text-danger">Failed to fetch answer.</div>`;
                }
                chatWindow.scrollTop = chatWindow.scrollHeight;
            }
        </script>
    </body>
    </html>
    """
print("✅ Safeguarded master main.py file successfully saved!")


Writing rag-agent/app/main.py


In [10]:
%%writefile rag-agent/requirements.txt
fastapi==0.110.0
uvicorn==0.28.0
groq==0.4.2
pypdf==4.1.0
fastembed==0.2.2
faiss-cpu==1.8.0
python-multipart==0.0.9
pydantic==2.6.4
numpy==1.26.4


Writing rag-agent/requirements.txt


In [11]:
%%writefile rag-agent/Dockerfile
FROM python:3.10-slim

WORKDIR /workspace

# Install system utilities needed for building packages cleanly
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 10000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "10000"]


Writing rag-agent/Dockerfile


In [12]:
%%writefile rag-agent/.dockerignore
*.pyc
__pycache__
.ipynb_checkpoints
*.ipynb


Writing rag-agent/.dockerignore


In [13]:
import threading
import uvicorn
from app.main import app
from google.colab import output

# Run the FastAPI server in a background thread so it doesn't lock up your notebook
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=10000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()

# Wait 2 seconds for the server boot cycle to initialize
import time
time.sleep(2)

print("🚀 FastAPI local test server is successfully running!")


ModuleNotFoundError: No module named 'app'

In [14]:
import os
import sys
import threading
import uvicorn
from google.colab import output

# Ensure the root path directory is explicitly registered
if os.path.abspath("rag-agent") not in sys.path:
    sys.path.append(os.path.abspath("rag-agent"))

# Import directly from the absolute package location path
from app.main import app

def run_server():
    # Use a clean loop framework to boot the Uvicorn application thread
    uvicorn.run(app, host="127.0.0.1", port=10000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()

import time
time.sleep(2)
print("🚀 FastAPI local test server is successfully running with path routing!")


✅ System escalation prompt file saved!
✅ Safeguarded master main.py file successfully saved!
🚀 FastAPI local test server is successfully running with path routing!


In [15]:
from google.colab import output

# Use Colab's iframe tool to display your actual web dashboard visually
output.serve_kernel_port_as_iframe(10000, height="600")


<IPython.core.display.Javascript object>

In [16]:
import shutil
from google.colab import files

# 1. Zip the clean folder tree
shutil.make_archive("rag_agent_deployment", "zip", "rag-agent")
print("✅ rag_agent_deployment.zip generated!")

# 2. Trigger an automatic file download to your computer
files.download("rag_agent_deployment.zip")


✅ rag_agent_deployment.zip generated!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>